# Task 27 - Lasso and Ridge Regression: tuning, shrinkage and coefficient behaviour

**Goal:** train Lasso and Ridge, compare them, tune the regularization strength systematically, assess feature shrinkage and visualise coefficient distributions.

**Target:** `log_revenue = log1p(price x units_sold)` (no `yearly amount spent` column exists; see Task 25).

**Features:** `FEATURE_SET = "pool"` (default) uses **all** pool features plus the interaction / square candidates from Task 25, where regularization matters most; `"selected"` uses only the final Task 25 features.

**Tuning:** 60 alphas on a log grid, 5-fold x 3 repeats; the **best alpha** (lowest CV RMSE) and the **1-SE alpha** (strongest regularization within one standard error of the best) are both reported. Performance is then estimated with **nested CV** (inner loop tunes alpha, outer loop scores) so the comparison is not optimistic. Elastic Net is added as a bridge between the two.
**Shrinkage analysis:** coefficient paths, number of non-zero coefficients vs alpha, L1 / L2 norms relative to OLS, per-feature shrinkage factors, multicollinearity (VIF) vs shrinkage.
**Coefficient behaviour:** coefficient distributions per model, top-coefficient comparison, bootstrap stability and Lasso selection frequency.

**Input:** `data/processed/womens_clothing_regression_dataset.csv` + `reports/task_25_selected_features.csv` (run Task 25 first)
**Outputs:** `reports/task_27_alpha_tuning.csv`, `reports/task_27_model_comparison.csv`, `reports/task_27_coefficients.csv`, `reports/task_27_shrinkage_summary.csv`, `reports/task_27_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

# Tasks 22-24 outputs sit in front of everything.
LATE2 = ["womens_clothing_rating_sentiment_encoded.csv", "womens_clothing_pca_features.csv"]
ALL = LATE2 + FULL

REG_DATASET = "womens_clothing_regression_dataset.csv"      # written by Task 25, read by Tasks 26-27

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, RepeatedKFold, cross_validate, cross_val_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42; TARGET_NAME = "log_revenue"
FEATURE_SET = "pool"                 # "pool" or "selected"
ALPHAS = np.logspace(-3, 2, 60)

## 1. Load data

In [ ]:
reg = pd.read_csv(PROC / "womens_clothing_regression_dataset.csv"); ft = pd.read_csv(REPORTS / "task_25_selected_features.csv")
y = reg[TARGET_NAME]
if FEATURE_SET == "selected": features = list(ft.loc[ft["selected"] | ft["interaction_kept"], "feature"])
else: features = list(ft["feature"])                                           # whole pool + interaction candidates
X = reg[features]; scaler = StandardScaler().fit(X); Xs = pd.DataFrame(scaler.transform(X), index=X.index, columns=features)
print(f"{len(X):,} rows x {X.shape[1]} features ({FEATURE_SET}) | condition number of the standardised design: {np.linalg.cond(Xs.values):.0f}")

## 2. Systematic alpha tuning (validation curves)

In [ ]:
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
def curve(make):
    rows = []
    for a in ALPHAS:
        s = -cross_val_score(make(a), X, y, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
        rows.append({"alpha": a, "rmse": s.mean(), "rmse_se": s.std() / np.sqrt(len(s))})
    return pd.DataFrame(rows)
ridge_c = curve(lambda a: make_pipeline(StandardScaler(), Ridge(alpha=a * 100)))     # ridge needs larger alphas for the same shrinkage; x100 keeps one shared grid
lasso_c = curve(lambda a: make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=20000)))
ols_rmse = -cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X, y, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()

def pick_alpha(c):
    b = c.loc[c["rmse"].idxmin()]; one_se = c[(c["rmse"] <= b["rmse"] + b["rmse_se"])]["alpha"].max()
    return float(b["alpha"]), float(one_se), float(b["rmse"])
ridge_best, ridge_1se, ridge_rmse = pick_alpha(ridge_c); lasso_best, lasso_1se, lasso_rmse = pick_alpha(lasso_c)
print(f"Ridge  best alpha {ridge_best * 100:.3g} (CV RMSE {ridge_rmse:.4f}), 1-SE alpha {ridge_1se * 100:.3g}\nLasso  best alpha {lasso_best:.3g} (CV RMSE {lasso_rmse:.4f}), 1-SE alpha {lasso_1se:.3g}\nOLS CV RMSE {ols_rmse:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for a_, c, nm, mult, b, s1 in [(ax[0], ridge_c, "Ridge", 100, ridge_best, ridge_1se), (ax[1], lasso_c, "Lasso", 1, lasso_best, lasso_1se)]:
    a_.errorbar(c["alpha"] * mult, c["rmse"], yerr=c["rmse_se"], fmt="-o", ms=3); a_.axhline(ols_rmse, color="k", ls="--", label="OLS")
    a_.axvline(b * mult, color="g", ls=":", label="best"); a_.axvline(s1 * mult, color="r", ls=":", label="1-SE"); a_.set_xscale("log"); a_.set_xlabel("alpha"); a_.set_ylabel("CV RMSE"); a_.set_title(f"{nm}: validation curve"); a_.legend()
plt.tight_layout(); plt.show()
tuning = pd.concat([ridge_c.assign(model="Ridge", alpha_used=ridge_c["alpha"] * 100), lasso_c.assign(model="Lasso", alpha_used=lasso_c["alpha"])], ignore_index=True)

## 3. Shrinkage: coefficient paths and sparsity

In [ ]:
Y = y.values - y.values.mean()
ridge_path = np.array([Ridge(alpha=a * 100).fit(Xs, y).coef_ for a in ALPHAS])
la, lcoefs, _ = lasso_path(Xs.values, Y, alphas=ALPHAS[::-1]); lasso_p = lcoefs.T[::-1]                    # (n_alphas, p) in the ALPHAS order
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
ax[0].plot(ALPHAS * 100, ridge_path, lw=.8); ax[0].set_xscale("log"); ax[0].axvline(ridge_best * 100, color="k", ls=":"); ax[0].set_title("Ridge coefficient paths"); ax[0].set_xlabel("alpha")
ax[1].plot(ALPHAS, lasso_p, lw=.8); ax[1].set_xscale("log"); ax[1].axvline(lasso_best, color="k", ls=":"); ax[1].set_title("Lasso coefficient paths"); ax[1].set_xlabel("alpha")
nz = (np.abs(lasso_p) > 1e-8).sum(axis=1); ax[2].plot(ALPHAS, nz, marker="o", ms=3); ax[2].set_xscale("log"); ax[2].axvline(lasso_best, color="k", ls=":"); ax[2].axvline(lasso_1se, color="r", ls=":"); ax[2].set_title("Lasso: non-zero coefficients vs alpha"); ax[2].set_xlabel("alpha")
plt.tight_layout(); plt.show()

## 4. Fit the final models and compare coefficients

In [ ]:
ols = LinearRegression().fit(Xs, y)
models = {"OLS": ols, "Ridge (best)": Ridge(alpha=ridge_best * 100).fit(Xs, y), "Ridge (1-SE)": Ridge(alpha=ridge_1se * 100).fit(Xs, y),
          "Lasso (best)": Lasso(alpha=lasso_best, max_iter=50000).fit(Xs, y), "Lasso (1-SE)": Lasso(alpha=lasso_1se, max_iter=50000).fit(Xs, y)}
enet_cv = ElasticNetCV(l1_ratio=[.2, .5, .8, .95], alphas=ALPHAS, cv=5, random_state=RANDOM_STATE, max_iter=20000).fit(Xs, y)
models["ElasticNet (CV)"] = ElasticNet(alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_, max_iter=50000).fit(Xs, y)
print(f"ElasticNet chose alpha={enet_cv.alpha_:.4g}, l1_ratio={enet_cv.l1_ratio_}")
coefs = pd.DataFrame({k: m.coef_ for k, m in models.items()}, index=features)
vif = pd.Series([variance_inflation_factor(Xs.values, i) for i in range(Xs.shape[1])], index=features, name="vif")
coefs["vif"] = vif
sum_rows = []
for k, m in models.items():
    c = m.coef_; sum_rows.append({"model": k, "nonzero_coefficients": int((np.abs(c) > 1e-8).sum()), "l1_norm": np.abs(c).sum(), "l2_norm": np.sqrt((c ** 2).sum()),
                                  "l2_vs_ols": np.sqrt((c ** 2).sum()) / np.sqrt((ols.coef_ ** 2).sum()), "max_abs_coef": np.abs(c).max(),
                                  "corr_with_ols_coef": np.corrcoef(c, ols.coef_)[0, 1] if k != "OLS" else 1.0})
shrink = pd.DataFrame(sum_rows); shrink.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
for k in ["OLS", "Ridge (best)", "Lasso (best)"]: sns.kdeplot(coefs[k], ax=ax[0], label=k, fill=True, alpha=.25)
ax[0].set_title("Distribution of coefficients"); ax[0].legend(); ax[0].set_xlabel("standardised coefficient")
sns.boxplot(data=coefs[["OLS", "Ridge (best)", "Ridge (1-SE)", "Lasso (best)", "Lasso (1-SE)", "ElasticNet (CV)"]], ax=ax[1], orient="h"); ax[1].set_title("Coefficient spread by model")
ax[2].scatter(coefs["OLS"], coefs["Ridge (best)"], s=14, alpha=.6, label="Ridge"); ax[2].scatter(coefs["OLS"], coefs["Lasso (best)"], s=14, alpha=.6, label="Lasso")
lim = np.abs(coefs["OLS"]).max(); ax[2].plot([-lim, lim], [-lim, lim], "k--", lw=.8); ax[2].set_xlabel("OLS coefficient"); ax[2].set_ylabel("regularised coefficient"); ax[2].legend(); ax[2].set_title("Shrinkage toward zero")
plt.tight_layout(); plt.show()

In [ ]:
top = coefs.drop(columns="vif").abs().max(axis=1).sort_values(ascending=False).index[:20]
coefs.loc[top[::-1], ["OLS", "Ridge (best)", "Lasso (best)", "ElasticNet (CV)"]].plot.barh(figsize=(10, 9), width=.8); plt.axvline(0, color="k", lw=.8); plt.title("Top-20 coefficients across models"); plt.tight_layout(); plt.show()
# does multicollinearity drive shrinkage? Ridge shrinks high-VIF features more
sf = (coefs["Ridge (best)"].abs() / coefs["OLS"].abs().replace(0, np.nan)).clip(upper=2)
plt.figure(figsize=(7, 4.5)); plt.scatter(coefs["vif"], sf, s=14); plt.xscale("log"); plt.xlabel("VIF"); plt.ylabel("|Ridge| / |OLS| (shrinkage factor)"); plt.title("Collinear features are shrunk more"); plt.show()
print("Spearman(VIF, shrinkage factor):", round(pd.Series(coefs["vif"]).corr(sf, method="spearman"), 3))

## 5. Nested cross-validation (unbiased comparison)

The inner loop picks alpha (RidgeCV / LassoCV / ElasticNetCV), the outer 5-fold x 3 repeats loop measures RMSE, MAE and R2 on data that never influenced the choice.

In [ ]:
nest = {"OLS": make_pipeline(StandardScaler(), LinearRegression()),
        "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS * 100, cv=5)),
        "Lasso": make_pipeline(StandardScaler(), LassoCV(alphas=ALPHAS, cv=5, max_iter=20000, random_state=RANDOM_STATE)),
        "ElasticNet": make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[.2, .5, .8], alphas=ALPHAS[::2], cv=5, max_iter=20000, random_state=RANDOM_STATE))}
rows = []
for n, m in nest.items():
    r = cross_validate(m, X, y, cv=cv, scoring={"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}, return_train_score=True, n_jobs=-1)
    rows.append({"model": n, "rmse": -r["test_rmse"].mean(), "rmse_sd": r["test_rmse"].std(), "mae": -r["test_mae"].mean(), "r2": r["test_r2"].mean(), "r2_sd": r["test_r2"].std(), "train_r2": r["train_r2"].mean()})
comp = pd.DataFrame(rows); comp["overfit_gap_r2"] = comp["train_r2"] - comp["r2"]; comp.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, m, t in zip(ax, ["rmse", "mae", "r2"], ["Nested CV RMSE", "Nested CV MAE", "Nested CV R2"]): sns.barplot(data=comp, x="model", y=m, ax=a, color="#4c72b0"); a.set_title(t); a.set_ylim(comp[m].min() * (.97 if m != "r2" else 0.9 if comp[m].min() > 0 else 1.1), comp[m].max() * 1.03 if comp[m].max() > 0 else 0.05)
plt.tight_layout(); plt.show()

## 6. Stability: bootstrap coefficient distributions and Lasso selection frequency

In [ ]:
rng = np.random.default_rng(RANDOM_STATE); B = 150; boot = {"OLS": [], "Ridge": [], "Lasso": []}; selected_ct = np.zeros(len(features))
for _ in range(B):
    i = rng.integers(0, len(Xs), len(Xs)); xb, yb = Xs.values[i], y.values[i]
    boot["OLS"].append(LinearRegression().fit(xb, yb).coef_); boot["Ridge"].append(Ridge(alpha=ridge_best * 100).fit(xb, yb).coef_)
    lc = Lasso(alpha=lasso_best, max_iter=20000).fit(xb, yb).coef_; boot["Lasso"].append(lc); selected_ct += np.abs(lc) > 1e-8
boot = {k: np.array(v) for k, v in boot.items()}; sel_freq = pd.Series(selected_ct / B, index=features, name="lasso_selection_frequency")
spread = pd.DataFrame({k: v.std(axis=0) for k, v in boot.items()}, index=features); spread.mean().round(4).to_frame("mean bootstrap SD of coefficients")

In [ ]:
t6 = coefs.drop(columns="vif").abs().max(axis=1).sort_values(ascending=False).index[:8]
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
rows = [pd.DataFrame({"feature": f, "model": k, "coef": boot[k][:, features.index(f)]}) for f in t6 for k in boot]
sns.boxplot(data=pd.concat(rows), x="coef", y="feature", hue="model", ax=ax[0], showfliers=False); ax[0].axvline(0, color="k", lw=.8); ax[0].set_title("Bootstrap coefficient distributions (top-8 features)")
sel_freq.sort_values(ascending=False).head(25).iloc[::-1].plot.barh(ax=ax[1], color="#55a868"); ax[1].set_title("Lasso selection frequency across bootstraps"); plt.tight_layout(); plt.show()
print(f"Features chosen in >= 80% of bootstraps: {(sel_freq >= .8).sum()} | in 20-80%: {((sel_freq >= .2) & (sel_freq < .8)).sum()} | in < 20%: {(sel_freq < .2).sum()}")

## 7. Save outputs

In [ ]:
coefs_out = coefs.join(sel_freq).join(spread.add_prefix("boot_sd_")); coefs_out.round(6).reset_index(names="feature").to_csv(REPORTS / "task_27_coefficients.csv", index=False)
tuning.round(6).to_csv(REPORTS / "task_27_alpha_tuning.csv", index=False); comp.round(6).to_csv(REPORTS / "task_27_model_comparison.csv", index=False)
shrink.assign(ridge_best_alpha=ridge_best * 100, ridge_1se_alpha=ridge_1se * 100, lasso_best_alpha=lasso_best, lasso_1se_alpha=lasso_1se).round(6).to_csv(REPORTS / "task_27_shrinkage_summary.csv", index=False)
best_model = comp.sort_values("rmse").iloc[0]
status = pd.DataFrame([{"task": "Task 27 - Lasso and Ridge regression", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"), "feature_set": FEATURE_SET,
    "n_features": len(features), "ridge_best_alpha": round(ridge_best * 100, 5), "lasso_best_alpha": round(lasso_best, 5), "lasso_nonzero_best": int(shrink.loc[shrink["model"] == "Lasso (best)", "nonzero_coefficients"].iloc[0]),
    "lasso_nonzero_1se": int(shrink.loc[shrink["model"] == "Lasso (1-SE)", "nonzero_coefficients"].iloc[0]), "nested_cv_rmse_ols": round(comp.loc[0, "rmse"], 4),
    "nested_cv_rmse_ridge": round(comp.loc[1, "rmse"], 4), "nested_cv_rmse_lasso": round(comp.loc[2, "rmse"], 4), "best_model": best_model["model"], "best_r2": round(best_model["r2"], 4)}])
status.to_csv(REPORTS / "task_27_analysis_status.csv", index=False); status.T

## 8. Key takeaways (fill in after running)

- Did regularization beat plain OLS in nested CV? With ~1.5k rows and a leakage-safe pool the gain is often small; what it buys is **stability** (lower coefficient spread) and **sparsity** (Lasso).
- How many coefficients does Lasso zero out at the best vs the 1-SE alpha, and are those features also the weak ones in the bootstrap selection frequency?
- Ridge shrinks correlated features smoothly (check the VIF plot), whereas Lasso picks one of a correlated group arbitrarily, so a feature dropped by Lasso is not necessarily unimportant.
- Coefficients are on standardised features and a log target: a coefficient of 0.1 means about a 10% change in expected revenue per one standard deviation of the feature.